In [5]:
# ========================================
# GRAD-CAM - Visual Explanation
# ========================================

import torch
import torch.nn.functional as F
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Enregistrer les hooks
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_backward_hook(self.save_gradient)
    
    def save_activation(self, module, input, output):
        self.activations = output.detach()
    
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    
    def generate_cam(self, input_tensor, target_class=None):
        """Générer la CAM"""
        # Forward pass
        logits = self.model(input_tensor)
        
        # Si pas de classe spécifiée, utiliser la classe prédite
        if target_class is None:
            target_class = logits.argmax(dim=1).item()
        
        # Backward pass
        self.model.zero_grad()
        loss = logits[:, target_class]
        loss.backward(torch.ones_like(loss))
        
        # Calculer les poids
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)
        cam = (weights * self.activations).sum(dim=1)  # (1, H, W)
        cam = F.relu(cam)
        cam = cam.squeeze().cpu().numpy()
        
        # Normaliser
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, logits.detach().cpu().numpy()[0]

def visualize_gradcam(images, labels, class_names, model, device, num_samples=12):
    """Visualiser Grad-CAM pour différentes images"""
    # Récupérer la dernière couche convolutional (layer4[-1])
    target_layer = model.layer4[-1]
    gradcam = GradCAM(model, target_layer)
    
    model.eval()
    
    fig, axes = plt.subplots(3, 4, figsize=(18, 14))
    axes = axes.ravel()
    
    for idx in range(min(num_samples, len(images))):
        img = images[idx].unsqueeze(0).to(device)
        
        with torch.no_grad():
            cam, logits = gradcam.generate_cam(img)
        
        # Dénormaliser l'image
        img_np = images[idx].cpu().numpy().transpose(1, 2, 0)
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())
        
        # Redimensionner CAM à la taille de l'image
        cam_resized = cv2.resize(cam, (img_np.shape[1], img_np.shape[0]))
        
        # Appliquer colormap
        heatmap = cv2.applyColorMap((cam_resized * 255).astype('uint8'), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
        
        # Superposer
        overlay = 0.6 * img_np + 0.4 * heatmap
        
        # Afficher
        axes[idx].imshow(overlay)
        true_label = class_names[labels[idx]]
        pred_label = class_names[logits.argmax()]
        confidence = logits.max()
        
        title = f'True: {true_label}\nPred: {pred_label} ({confidence:.2%})'
        color = 'green' if true_label == pred_label else 'red'
        axes[idx].set_title(title, color=color, fontweight='bold')
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig('gradcam_visualization.png', dpi=300, bbox_inches='tight')
    plt.show()

# UTILISATION
# visualize_gradcam(test_images, test_labels, class_names, model, device)